# Census Example — TabPFN (GPU run)

This notebook produces the **TabPFN rows of Tables 8 and 9** (U.S. labor force, Appendix B.2)
of *"The choice of reference group can reverse conclusions in the Oaxaca–Blinder decomposition."*
It is the census counterpart of `Real-data example/icu_139_tabpfn_local.ipynb` and is meant to be
run on a GPU node; everything else in Tables 8–9 comes from `census/py/` on CPU.

It reuses the shared engine (`obd_engine/`), so the subsets, features, decomposition, bootstrap,
and flip counting are *identical* to the OLS / logistic / XGBoost / NN runs — only the model
differs. Same TabPFN setup as the ICU run: v2.6 weights, `tabpfn` 7.1 code,
`ignore_pretraining_limits=True`.

## What it produces

| Phase | Output | Notes |
|---|---|---|
| 1 — point estimates on all cells | `census/temp/nonlinear_fits_tabpfn.parquet` | resumable per cell (`census/temp/tabpfn_phase1/`) |
| 2 — bootstrap, **B = 1000**, flip cells only | `census/temp/nonlinear_boots_tabpfn.parquet` | resumable per cell, via `02_bootstrap_flips.py` |
| 3 — flip counts in Tables 8/9 format | `census/out_py/flip_counts_tabpfn.csv` | via `03_count_flips.py` |

## One-time setup on the node

```
pip install -r census/py/requirements.txt tabpfn==7.1.1
```

**Weights.** TabPFN ≥ 7 downloads checkpoints only after the license is accepted and
`TABPFN_TOKEN` is set (https://ux.priorlabs.ai). Two checkpoints are needed here — the
**classifier** (health-insurance outcome) *and* the **regressor** (log income). On an
air-gapped node copy both v2.6 `.ckpt` files over and set `TABPFN_CLF_PATH` / `TABPFN_REG_PATH`
in the config cell.

**Data.** The analysis table is shipped as `census/py/data/acs16_workforce.parquet`
(871,849 rows; derived from the public 2016 ACS 1-Year PUMS). To rebuild it from the raw
files instead, follow the README's census instructions and run `python census/py/00_load_acs.py`.

## Configuration (next cell)

- `TABPFN_DEVICE` — `"cuda"` on the GPU node (`"cpu"` works but is very slow).
- `TABPFN_MAX_ROWS` — TabPFN's context is bounded; groups larger than this are **fit on a
  fixed random subsample** of this many rows but **predicted on every row**, so the
  counterfactual means still average over the whole group. 10 000 is safe on an 80 GB A100;
  raise it if memory allows (cost grows roughly quadratically).
- `B_BOOT` — 1000, matching the paper. Phase 1 prints per-cell timings so you can extrapolate
  Phase 2's cost before committing to it (rule of thumb: ~90 flip cells × 2 × B fits).
- `TEST_MODE` — `True` runs one small cell end to end (Alaska, women vs. men, insurance) with
  a tiny bootstrap; use it to check the environment, then set it to `False`.

Run every cell top to bottom. Re-running after an interruption skips finished cells.

In [ ]:
# ============================================================
# SECTION 1 — Config  (EDIT THIS SECTION for your machine)
# ============================================================
import os, sys, time, json, subprocess, re
from pathlib import Path

TEST_MODE = False               # True: one small cell + B=5, to check the environment

TABPFN_DEVICE   = "cuda"        # "cuda" | "cpu" | "mps"(opt-in; segfaults with some torch builds)
TABPFN_CLF_PATH = None          # e.g. "/home/me/tabpfn-v2.6-classifier-v2.6_default.ckpt"; None = TabPFN cache/download
TABPFN_REG_PATH = None          # e.g. "/home/me/tabpfn-v2.6-regressor-v2.6_default.ckpt"
TABPFN_MAX_ROWS = 10_000        # training rows per group (predictions use all rows)

B_BOOT      = 1000              # bootstrap replicates (paper: 1000)
N_JOBS_BOOT = 1                 # replicates in parallel: 1 per GPU
ALPHAS      = [0.10, 0.05, 0.01]

# Design grid -- identical to census/py/01_fit_decomposition.py
SUBSET_COLS = ["st", "naics_2"]
POP_NAMES   = ["sex_female", "race_bw", "immigrant"]
OUTCOMES    = ["pincp", "hicov"]          # log income (regressor), insurance (classifier)
ALGO        = "tabpfn"
OUT_SUFFIX  = "_tabpfn"

if TEST_MODE:
    SUBSET_COLS, POP_NAMES, OUTCOMES, B_BOOT = ["st"], ["sex_female"], ["hicov"], 5
    OUT_SUFFIX = "_tabpfn_test"

# The engine reads these from the environment (obd_engine/builders.py), so set them
# BEFORE importing it. They are also passed to the 02/03 scripts in Phases 2-3.
ENV = {
    "OBD_TABPFN_DEVICE": TABPFN_DEVICE,
    "OBD_TABPFN_MAX_ROWS": str(TABPFN_MAX_ROWS),
    "OBD_OUT_SUFFIX": OUT_SUFFIX,
    "OBD_B": str(B_BOOT),
    "OBD_INNER_JOBS": str(N_JOBS_BOOT),
    "OBD_BOOT_ALGOS": ALGO,
    "OBD_N_JOBS": "1",
}
if TABPFN_CLF_PATH: ENV["OBD_TABPFN_CLF_PATH"] = TABPFN_CLF_PATH
if TABPFN_REG_PATH: ENV["OBD_TABPFN_REG_PATH"] = TABPFN_REG_PATH
os.environ.update(ENV)

ROOT = Path.cwd()
while not (ROOT / "obd_engine").is_dir():          # notebook may be opened from census/py or the root
    if ROOT.parent == ROOT: raise FileNotFoundError("run from inside the ReferenceOaxacaBlinder repo")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from obd_engine import decompose_from_data
print(f"repo root: {ROOT}\nTEST_MODE={TEST_MODE}  device={TABPFN_DEVICE}  max_rows={TABPFN_MAX_ROWS}  B={B_BOOT}  suffix={OUT_SUFFIX}")

## Section 2 — Data and design grid

Same restriction, subsets, and groups as `01_fit_decomposition.py`. Rows with a missing group
value (`race_bw` is NA for non-Black/non-White respondents) are dropped only when that
column is the group being compared.

In [ ]:
# ============================================================
# SECTION 2 — Load data, build the design grid
# ============================================================
DATA_CANDIDATES = [ROOT / "census/temp/acs16_workforce.parquet", ROOT / "census/py/data/acs16_workforce.parquet"]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "acs16_workforce.parquet not found: run census/py/00_load_acs.py or copy the shipped file"
acs = pd.read_parquet(DATA_PATH).drop(columns=["naics_3"])
(ROOT / "census/temp").mkdir(exist_ok=True)
print(f"{DATA_PATH.relative_to(ROOT)}: {len(acs):,} rows")

subsets = (acs[SUBSET_COLS].astype(str).melt(var_name="subset_name", value_name="subset_value")
             .drop_duplicates().sort_values(["subset_name", "subset_value"]).reset_index(drop=True))
if TEST_MODE:
    subsets = subsets[subsets["subset_value"] == "2"]   # Alaska, the smallest state (n = 1,384)

KEYS = ["subset_name", "subset_value", "pop_name", "y_name", "algo_name"]
design = pd.DataFrame([dict(subset_name=r.subset_name, subset_value=r.subset_value, pop_name=p, y_name=y, algo_name=ALGO)
                       for r in subsets.itertuples(index=False) for p in POP_NAMES for y in OUTCOMES])
sizes = subsets.assign(n=[int((acs[r.subset_name].astype(str) == r.subset_value).sum()) for r in subsets.itertuples(index=False)])
print(f"{len(subsets)} subsets x {len(POP_NAMES)} groups x {len(OUTCOMES)} outcomes = {len(design)} cells; "
      f"subset sizes {sizes.n.min():,} .. {sizes.n.max():,} rows")

## Phase 1 — TabPFN point estimates on every cell (resumable)

One classifier or regressor per group per cell (two fits), then the four counterfactual
means. Each finished cell is cached as a CSV so an interrupted run resumes where it stopped.
The per-cell timing printed here is the number to extrapolate Phase 2 from.

In [ ]:
# ============================================================
# PHASE 1 — point estimates, cached per cell
# ============================================================
def cell_slug(cell) -> str:
    raw = "__".join(str(getattr(cell, k)) for k in KEYS)
    return re.sub(r"[^A-Za-z0-9_.=-]+", "-", raw)

P1_DIR = ROOT / "census/temp" / f"tabpfn_phase1{OUT_SUFFIX}"; P1_DIR.mkdir(parents=True, exist_ok=True)
rows, t_all = [], time.time()
for i, cell in enumerate(design.itertuples(index=False), start=1):
    cache = P1_DIR / f"{cell_slug(cell)}.csv"
    if cache.exists():
        rows.append(pd.read_csv(cache).iloc[0].to_dict()); continue
    sub = acs[acs[cell.subset_name].astype(str) == cell.subset_value]
    t = time.time()
    res = decompose_from_data(sub, cell.y_name, cell.pop_name, ALGO)
    row = {k: getattr(cell, k) for k in KEYS} | res | {"seconds": round(time.time() - t, 1)}
    pd.DataFrame([row]).to_csv(cache, index=False); rows.append(row)
    flip = "FLIP" if (res["explained_0"] * res["explained_1"] < 0 or res["unexplained_0"] * res["unexplained_1"] < 0) else ""
    print(f"  [{i}/{len(design)}] {cell.subset_name}={cell.subset_value:<4} {cell.pop_name:<10} {cell.y_name}  "
          f"n={res['n_0'] + res['n_1']:>6,}  {row['seconds']:6.1f}s  {flip}", flush=True)

fits = pd.DataFrame(rows)
fits_path = ROOT / "census/temp" / f"nonlinear_fits{OUT_SUFFIX}.parquet"
fits.drop(columns=["seconds"]).to_parquet(fits_path, index=False)          # 01's schema, consumed by 02/03
fits.to_csv(ROOT / "census/out_py" / f"tabpfn_phase1_fits{OUT_SUFFIX}.csv", index=False)
print(f"\nPhase 1 done: {len(fits)} cells, {fits['seconds'].sum() / 60:.1f} GPU-minutes of fitting -> {fits_path.name}")

In [ ]:
# Point-estimate flip counts and a Phase-2 cost estimate (same filter as 02/03)
sized = fits[(fits.n_0 > 50) & (fits.n_1 > 50) & (fits.delta_y.abs() > 0.01)]
e = sized.explained_0 * sized.explained_1 < 0; u = sized.unexplained_0 * sized.unexplained_1 < 0
summary = (sized.assign(exp_flip=e, unexp_flip=u, any_flip=e | u)
                .groupby("y_name")[["any_flip", "exp_flip", "unexp_flip"]].sum().astype(int)
                .join(sized.groupby("y_name").size().rename("n_fit")))
display(summary)
flip_cells = sized[e | u]
est_hours = (flip_cells["seconds"].sum() * B_BOOT) / 3600
print(f"Phase 2 will bootstrap {len(flip_cells)} flip cells x B={B_BOOT}: about {est_hours:,.0f} GPU-hours "
      f"at the Phase-1 per-cell rate (each replicate re-fits both group models).")

## Phase 2 — Bootstrap, B = 1000, flip cells only (resumable)

Runs `census/py/02_bootstrap_flips.py` on the TabPFN fits: within-group resampling with
replacement, both group models re-fit per replicate, one chunk file per cell in
`census/temp/nonlinear_boots_tabpfn/`. Interrupt and re-run freely — finished cells are skipped.

In [ ]:
# ============================================================
# PHASE 2 — bootstrap via the shared pipeline script
# ============================================================
def run_step(script):
    print(f"$ {' '.join(f'{k}={v}' for k, v in ENV.items())} python {script}", flush=True)
    proc = subprocess.run([sys.executable, str(ROOT / "census/py" / script)], env={**os.environ, **ENV},
                          cwd=ROOT, text=True, capture_output=True)
    print(proc.stdout[-4000:]);
    if proc.returncode != 0:
        print(proc.stderr[-4000:]); raise RuntimeError(f"{script} failed")

run_step("02_bootstrap_flips.py")

## Phase 3 — Flip counts in Tables 8 / 9 format

`census/py/03_count_flips.py`: bootstrap SEs, normal-approximation p-values, and the counts —
`is_flip` = point-estimate flips (the "Total" column), `reject_10/05/01` = flips where at least
one reference is significant at that level. `n_with_se` should equal `is_flip`; if it is
lower, some flip cells have no bootstrap yet.

In [ ]:
# ============================================================
# PHASE 3 — significance + flip-count rows
# ============================================================
run_step("03_count_flips.py")
counts = pd.read_csv(ROOT / "census/out_py" / f"flip_counts{OUT_SUFFIX}.csv")
counts = counts[counts.algo_name == ALGO]
print("Rows for Tables 8 (pincp) and 9 (hicov): component = either | explained | unexplained")
display(counts.pivot_table(index=["y_name", "component"], values=["n_fit", "is_flip", "reject_10", "reject_05", "reject_01", "n_with_se"], aggfunc="first")
              [["n_fit", "is_flip", "reject_10", "reject_05", "reject_01", "n_with_se"]])

## Hand-off

Send back `census/out_py/flip_counts_tabpfn.csv` and `tabpfn_phase1_fits_tabpfn.csv`. If Phase 2
had to stop early, also send `census/temp/nonlinear_boots_tabpfn/` (the per-cell chunks) so it
can be resumed elsewhere; `n_with_se` in the counts file shows how far it got.